# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from IPython.display import Markdown, display

display(Markdown("""
**My lane:** Refresh / Content Opportunity Scoring (same lane as Weeks 1-2).

**Unit of analysis:** One row = one content item (`content_hash_id`) belonging to one pseudonymized client (`client_hash_id`), aggregated over a fixed 30-day window inside a
single mid-panel month.

**Table(s):** `dim_content` (content metadata, joined for context) and `fact_content_daily_performance` (the daily x client x content fact table), filtered to
`month=2026-03` for all development in this notebook.

**Time window:** I split `month=2026-03`'s daily rows around its midpoint into a `prev30` window (earlier ~30 days — my features) and a `last30` window (later ~30 days — where my label is defined).
I never touch `fact_content_daily_performance_sample` here, since that table is exactly June 2026 (the final, sealed month) and would let a future-outcome label leak into a month I'm supposed to be predicting into.
"""))


**My lane:** Refresh / Content Opportunity Scoring (same lane as Weeks 1-2).

**Unit of analysis:** One row = one content item (`content_hash_id`) belonging to one pseudonymized client (`client_hash_id`), aggregated over a fixed 30-day window inside a
single mid-panel month.

**Table(s):** `dim_content` (content metadata, joined for context) and `fact_content_daily_performance` (the daily x client x content fact table), filtered to
`month=2026-03` for all development in this notebook.

**Time window:** I split `month=2026-03`'s daily rows around its midpoint into a `prev30` window (earlier ~30 days — my features) and a `last30` window (later ~30 days — where my label is defined).
I never touch `fact_content_daily_performance_sample` here, since that table is exactly June 2026 (the final, sealed month) and would let a future-outcome label leak into a month I'm supposed to be predicting into.


In [2]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort, never hardcode/paste in a cell).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): Please paste the actual token string, not Python code for retrieving it. ")

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
FACT_MONTH  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Connected. Tables pointed at month=2026-03 for the fact table.")

# NOTE (fix): a single month partition (31 days) cannot hold two real 30-day
# windows -- splitting inside March 2026 puts 30 days of impressions on one
# side and 1 day on the other, which silently breaks the label. Pull Feb+Mar
# so prev30/last30 sit on either side of a real calendar boundary.
FACT_TWO_MONTHS = (
    f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])"
)
print("Also connected FACT_TWO_MONTHS (Feb 2026 + Mar 2026) for the prev30/last30 label window.")

Paste your Hugging Face READ token (hf_...): Please paste the actual token string, not Python code for retrieving it. ··········
Connected. Tables pointed at month=2026-03 for the fact table.
Also connected FACT_TWO_MONTHS (Feb 2026 + Mar 2026) for the prev30/last30 label window.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
display(Markdown("""
| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | Pseudonymous IDs — grouping, joining, and client-holdout splits only, never model features |
| `report_date` | Context | Used to build the prev30/last30 windows, not fed to the model directly |
| `gsc_impressions` (prev30 sum) | Feature | Known before the decision point — it's the earlier window |
| `gsc_clicks` (prev30 sum) | Feature | Same — prior-window observed signal |
| `gsc_avg_position` (prev30 avg) | Feature | Same — prior-window observed signal |
| `word_count`, `content_type` (from `dim_content`) | Feature | Static content metadata, known well before any prediction moment |
| `gsc_impressions` (last30 sum) | Label input | This is where my label comes FROM (the outcome window) — never a feature |
| `is_declining` (derived: last30 < 0.8 * prev30) | Label | The target I'd rank/predict |
| `health_score`, `priority_score`, `action_type`, any refresh flag | Excluded | FlyRank product decisions, not observable signals — not present in this release anyway, and would be circular if they were |
| `ga4_data_available` | Context | A flag I filter on (`IS TRUE`), not a feature — tells me whether GA4 zeros are real zeros or missing tracking |
"""))


| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | Pseudonymous IDs — grouping, joining, and client-holdout splits only, never model features |
| `report_date` | Context | Used to build the prev30/last30 windows, not fed to the model directly |
| `gsc_impressions` (prev30 sum) | Feature | Known before the decision point — it's the earlier window |
| `gsc_clicks` (prev30 sum) | Feature | Same — prior-window observed signal |
| `gsc_avg_position` (prev30 avg) | Feature | Same — prior-window observed signal |
| `word_count`, `content_type` (from `dim_content`) | Feature | Static content metadata, known well before any prediction moment |
| `gsc_impressions` (last30 sum) | Label input | This is where my label comes FROM (the outcome window) — never a feature |
| `is_declining` (derived: last30 < 0.8 * prev30) | Label | The target I'd rank/predict |
| `health_score`, `priority_score`, `action_type`, any refresh flag | Excluded | FlyRank product decisions, not observable signals — not present in this release anyway, and would be circular if they were |
| `ga4_data_available` | Context | A flag I filter on (`IS TRUE`), not a feature — tells me whether GA4 zeros are real zeros or missing tracking |


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# --- Query A: grain check ---
# One row of the raw fact table really is one (report_date, client, content)?
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Grain violations (should be 0 rows):")
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be 0 rows):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [5]:
# --- Query B: row count + date span for this slice ---
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MONTH}
""").df()
print(span)

    n_rows  n_clients  n_content   min_date   max_date
0  9841378         55     331437 2026-03-01 2026-03-31


In [6]:
# --- Query C: availability — filtered honestly with IS TRUE / IS NOT TRUE ---
# ga4_data_available can be TRUE, FALSE, or NULL. '= FALSE' or 'NOT ...' would
# silently mishandle the NULLs, so IS TRUE / IS NOT TRUE is required here.
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)      AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END)  AS ga4_unavailable_or_null_rows
    FROM {FACT_MONTH}
""").df()
print(avail)
print(f"\nRows surviving an IS TRUE filter: {int(avail['ga4_available_rows'][0]):,} of {int(avail['total_rows'][0]):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  ga4_unavailable_or_null_rows
0     9841378            413966.0                     9427412.0

Rows surviving an IS TRUE filter: 413,966 of 9,841,378


In [7]:
# --- Build the prev30 / last30 feature-and-label frame (FIXED) ---
# Anchor the split at a real calendar boundary (March 1, 2026) across two
# month partitions, instead of splitting inside a single 31-day partition.
ANCHOR = "DATE '2026-03-01'"

features = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date < {ANCHOR}
                         THEN gsc_impressions ELSE 0 END)                        AS imp_prev30,
               SUM(CASE WHEN report_date < {ANCHOR}
                         THEN gsc_clicks ELSE 0 END)                             AS clk_prev30,
               AVG(CASE WHEN report_date < {ANCHOR}
                         THEN gsc_avg_position END)                              AS pos_prev30,
               SUM(CASE WHEN report_date >= {ANCHOR}
                         AND report_date < {ANCHOR} + INTERVAL 30 DAY
                         THEN gsc_impressions ELSE 0 END)                        AS imp_last30
        FROM {FACT_TWO_MONTHS}
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT a.*, d.word_count, d.content_type
    FROM agg a
    LEFT JOIN {DIM_CONTENT} d ON a.content_hash_id = d.content_hash_id
""").df()

features["is_declining"] = (features["imp_last30"] < 0.8 * features["imp_prev30"]).astype(int)
print(f"{len(features):,} content items with enough prev30 volume (imp_prev30 >= 100)")
print("Declining rate:", round(features["is_declining"].mean(), 3))
print("(Sanity check: this should now be well above 0 -- compare to the starter's own 0.542.)")
features.head()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

80,322 content items with enough prev30 volume (imp_prev30 >= 100)
Declining rate: 0.231
(Sanity check: this should now be well above 0 -- compare to the starter's own 0.542.)


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,imp_last30,word_count,content_type,is_declining
0,client_3ffa76342f366962,content_32bdebcb01540202,551.0,17.0,3.815812,323.0,904,feedly article,1
1,client_e547b89c05043229,content_d0fa1bbfbc10caf8,957.0,0.0,14.080144,1550.0,3065,keyword article,0
2,client_e547b89c05043229,content_4c1e972bec56132e,2882.0,15.0,10.396231,3023.0,2993,keyword article,0
3,client_e547b89c05043229,content_64cad58fc02e7605,549.0,0.0,30.741277,1262.0,3048,keyword article,0
4,client_e547b89c05043229,content_4e48bd81bb37eb4f,7343.0,3.0,44.960642,14860.0,<NA>,keyword article,0


In [8]:
# --- The trap: add ONE label-derived column on purpose, watch the score jump, then remove it ---
from sklearn.tree import DecisionTreeClassifier

X_cols = ["imp_prev30", "clk_prev30", "pos_prev30"]
X = features[X_cols].fillna(0)
y = features["is_declining"]

honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
honest_tree.fit(X, y)
print(f"Honest in-sample score (prev30-only features): {honest_tree.score(X, y):.3f}")

# Deliberately leak: imp_last30 is literally what the label is computed from.
X_leaky = features[X_cols + ["imp_last30"]].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
leaky_tree.fit(X_leaky, y)
print(f"'Leaky' in-sample score (imp_last30 included): {leaky_tree.score(X_leaky, y):.3f}  <- looks amazing, means nothing")

print("\nDeleting imp_last30 from the feature set now — it never belongs in a real model,")
print("since it is the outcome window the label itself is drawn from.")
X_final = X  # honest feature set going forward

Honest in-sample score (prev30-only features): 0.535
'Leaky' in-sample score (imp_last30 included): 0.869  <- looks amazing, means nothing

Deleting imp_last30 from the feature set now — it never belongs in a real model,
since it is the outcome window the label itself is drawn from.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
display(Markdown("""
**Named limitation for this slice:** `month=2026-03` is a single mid-panel month, and history depth differs sharply per client (`dim_clients.gsc_data_start` / `ga4_data_start`) — some clients have over a year of history behind this month, othersvery little.
I filtered `ga4_data_available IS TRUE` above, which means clients with a late `ga4_data_start` are under-represented in any GA4-derived feature, not because they have low engagement but because tracking hadn't started yet. `ga4_data_available` can
also be NULL (not just FALSE) for a subset of rows, so an `IS TRUE` filter is doing real work here, not a formality.

This data also cannot support any causal claim ("refreshing this page would recover
traffic") — it is observational, cross-sectional-within-a-window data, so results here
stay in observed / directional / decision-support language, never causal.

Finally, this is one month out of ~17 months of panel history — a single month's
prev30/last30 split is a reasonable prototype but not yet validated across other months
or client cohorts; that generalization check belongs to later weeks (ML-08/ML-09).
"""))


**Named limitation for this slice:** `month=2026-03` is a single mid-panel month, and history depth differs sharply per client (`dim_clients.gsc_data_start` / `ga4_data_start`) — some clients have over a year of history behind this month, othersvery little. 
I filtered `ga4_data_available IS TRUE` above, which means clients with a late `ga4_data_start` are under-represented in any GA4-derived feature, not because they have low engagement but because tracking hadn't started yet. `ga4_data_available` can
also be NULL (not just FALSE) for a subset of rows, so an `IS TRUE` filter is doing real work here, not a formality.

This data also cannot support any causal claim ("refreshing this page would recover
traffic") — it is observational, cross-sectional-within-a-window data, so results here
stay in observed / directional / decision-support language, never causal.

Finally, this is one month out of ~17 months of panel history — a single month's
prev30/last30 split is a reasonable prototype but not yet validated across other months
or client cohorts; that generalization check belongs to later weeks (ML-08/ML-09).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.